# Gemini API Latency Test
Test different Gemini models and thinking configurations.

In [7]:
import time
import os
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('GEMINI_API_KEY')
assert API_KEY, 'GEMINI_API_KEY not set in .env'
BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/models'

def call_gemini(model_id, prompt, temperature=0, thinking_config=None, system_instruction=None, timeout=120):
    """Call Gemini REST API and return (response_text, latency, usage_metadata, raw_data)."""
    url = f'{BASE_URL}/{model_id}:generateContent?key={API_KEY}'
    gen_config = {'temperature': temperature}
    if thinking_config:
        gen_config['thinkingConfig'] = thinking_config
    body = {
        'contents': [{'parts': [{'text': prompt}]}],
        'generationConfig': gen_config
    }
    if system_instruction:
        body['system_instruction'] = {'parts': [{'text': system_instruction}]}
    
    t0 = time.time()
    resp = requests.post(url, json=body, timeout=timeout)
    latency = round(time.time() - t0, 2)
    data = resp.json()
    
    if 'error' in data:
        return None, latency, {}, data
    
    usage = data.get('usageMetadata', {})
    text = data.get('candidates', [{}])[0].get('content', {}).get('parts', [{}])[0].get('text', '')
    return text, latency, usage, data

print('Ready!')

Ready!


In [8]:
PROMPT = 'Say hello in one sentence.'

call_gemini('gemini-2.0-flash', PROMPT, thinking_config=None)

("Hello there, I hope you're having a great day!\n",
 1248.7,
 {'promptTokenCount': 6,
  'candidatesTokenCount': 14,
  'totalTokenCount': 20,
  'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}],
  'candidatesTokensDetails': [{'modality': 'TEXT', 'tokenCount': 14}]},
 {'candidates': [{'content': {'parts': [{'text': "Hello there, I hope you're having a great day!\n"}],
     'role': 'model'},
    'finishReason': 'STOP',
    'avgLogprobs': -0.0767847214426313}],
  'usageMetadata': {'promptTokenCount': 6,
   'candidatesTokenCount': 14,
   'totalTokenCount': 20,
   'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}],
   'candidatesTokensDetails': [{'modality': 'TEXT', 'tokenCount': 14}]},
  'modelVersion': 'gemini-2.0-flash',
  'responseId': 'KK2Kacv2GL_ujrEPi8aW0AY'})

## Quick latency test across models

In [6]:
PROMPT = 'Say hello in one sentence.'

models = [
    ('gemini-2.0-flash', None),
    ('gemini-2.5-flash', {'thinkingBudget': 0}),
    ('gemini-2.5-flash', None),
    ('gemini-3-flash-preview', {'thinkingLevel': 'LOW'}),
]

for model_id, thinking in models:
    label = model_id + (f' {thinking}' if thinking else '')
    try:
        text, latency, usage, _ = call_gemini(model_id, PROMPT, thinking_config=thinking)
        if text:
            thinking_tokens = usage.get('thoughtsTokenCount', 0) or 0
            print(f'{label}: {latency}s | thinking={thinking_tokens} | "{text[:80]}"')
        else:
            print(f'{label}: ERROR in {latency}s')
    except Exception as e:
        print(f'{label}: {e}')

gemini-2.0-flash: 0.81s | thinking=0 | "Hello there, I hope you're having a great day!
"
gemini-2.5-flash {'thinkingBudget': 0}: 0.66s | thinking=0 | "Hello!"
gemini-2.5-flash: 1.18s | thinking=20 | "Hello there!"
gemini-3-flash-preview {'thinkingLevel': 'LOW'}: 2.76s | thinking=110 | "Hello, I hope you are having a wonderful day!"


## Custom test — edit prompt, model, thinking config

In [ ]:
# Edit these:
MODEL = 'gemini-2.5-flash'
PROMPT = 'Say hello in one sentence.'
THINKING = {'thinkingBudget': 0}  # or {'thinkingLevel': 'LOW'} or None
SYSTEM = None  # or 'You are a helpful assistant.'

text, latency, usage, raw = call_gemini(MODEL, PROMPT, thinking_config=THINKING, system_instruction=SYSTEM)
print(f'Model: {MODEL}')
print(f'Latency: {latency}s')
print(f'Usage: {usage}')
print(f'Response: {text}')

## Repeat test — run N times to check consistency

In [ ]:
MODEL = 'gemini-2.5-flash'
PROMPT = 'Say hello in one sentence.'
THINKING = {'thinkingBudget': 0}
N = 5

latencies = []
for i in range(N):
    text, latency, usage, _ = call_gemini(MODEL, PROMPT, thinking_config=THINKING)
    latencies.append(latency)
    print(f'Run {i+1}: {latency}s')

print(f'\nAvg: {sum(latencies)/len(latencies):.2f}s | Min: {min(latencies)}s | Max: {max(latencies)}s')